# From paper to practice: STH-LRTC for blanked traffic cells, and what a third mode (day of week) adds

**TrafficFlowBench, Task 1** blanks 20–50 % of the cells of each day, mostly in 90-minute bands, and asks for speed and flow there. This notebook
takes one published method for that kind of gap - *spatio-temporal Hankel low-rank tensor completion* (STH-LRTC; Wang, Wu, Zhuang & Sun 2021,
arXiv 2105.11335, building on the LRTC-TNN of Chen, Yang & Sun 2020) - and applies it **as it is written in the paper**, in about 60 lines of PyTorch,
with a self-test you can run here. Then it adds one thing the paper does not have, **the day of the week as a third mode**, and measures what that is worth,
on the data and on the leaderboard.

**The short version, with the numbers below in the notebook**
* The method works: on real corridor-days with hidden 90-minute blocks, STH per day cuts the speed error of plain time-interpolation by ~18 %, and the weekday-3-D variant by ~35 % (mean of 4 random blockings, 5 Tuesdays of one corridor; one blocking out of four goes the other way - see section 4).
* It is **not** uniformly the best fill: on six validation groups the older LRTC-TNN is best in 3, the weekday-3-D variant in 2, STH per day in 1 (section 5).
* **The best stand-alone fill was not the best contribution to the model.** Uploaded *as the answer* it scored −0.00034 below our base; as four extra columns in a gradient-boosted model it added +0.00008 (local, @2 %). We predicted the first number to within 0.00001 before submitting (section 6).
* What this is *not*: a new algorithm, or a leaderboard recipe. It is a worked example of taking a paper to a working, tested implementation, and of how little a good component can be worth in a stack.

In [ ]:
import os, glob, numpy as np, pandas as pd
from pathlib import Path
KIN = Path(os.environ.get('KIN', '/kaggle/input'))
root = [p for depth in range(1, 6) for p in KIN.glob('/'.join(['*'] * depth) + '/corridors') if (p / 'D12_I5_N').exists()][0]
def corridor(panel):
    """links in driving order, x_km along the direction of travel, free-flow speed and length per link"""
    topo = pd.read_csv(f'{root}/{panel}/network/lwr_mainline_topology.csv'); fd = pd.read_csv(f'{root}/{panel}/network/fd_parameters.csv').set_index('link_id')
    f = sorted(glob.glob(f'{root}/{panel}/train/mainline_states/*/synthetic_mainline_*.parquet'))[0]
    mp = pd.read_parquet(f, columns=['link_id', 'milepost']).groupby('link_id').milepost.mean(); ml = set(topo.link_id)
    s = [np.sign(mp[o] - mp[r.link_id]) for r in topo.itertuples() for o in str(r.outgoing_link_ids).split(';') if o in ml and r.link_id in mp.index and o in mp.index]
    sgn = 1.0 if np.sum(s) >= 0 else -1.0
    order = (mp[mp.index.isin(ml)] * sgn).sort_values().index.tolist(); x = mp.reindex(order).to_numpy() * sgn
    return dict(order=order, x=x - x.min(), vf=fd.loc[order, 'free_speed_kmh'].to_numpy(), length=fd.loc[order, 'length_km'].to_numpy(), cap=fd.loc[order, 'capacity_vph'].to_numpy(), lanes=fd.loc[order, 'lanes'].to_numpy())
def day_grid(panel, date, order, col='speed_kmh'):
    """(288, L) grid of one train day; NaN where the cell is not score-eligible (not observed)"""
    f = glob.glob(f'{root}/{panel}/train/mainline_states/*/synthetic_mainline_{date.replace("-", "_")}.parquet')[0]
    d = pd.read_parquet(f, columns=['timestamp', 'link_id', col, 'is_score_eligible']); d = d[d.is_score_eligible.astype(bool)]
    li = {l: i for i, l in enumerate(order)}; t = pd.to_datetime(d.timestamp); G = np.full((288, len(order)), np.nan)
    G[(t.dt.hour * 12 + t.dt.minute // 5).to_numpy(), d.link_id.map(li).to_numpy()] = d[col].to_numpy(); return G
import torch
from pathlib import Path
from IPython.display import Image, display
KIN = Path(os.environ.get('KIN', '/kaggle/input'))
def show(name, w=1000):
    hits = list(KIN.glob(f'**/{name}'))
    assert hits, f'figure {name} not found under {KIN} - attach the dataset plarmuseau/trafficflowbench-three-figs'
    display(Image(filename=str(hits[0]), width=w))
print('competition data at', root)

## 1. The problem in one picture

A day is a matrix: links (the road, in driving order) × 288 five-minute steps. Jams are the red wedges; free flow is green. The released day
has holes - cells that were never observed, plus the blocks the task blanks. The baseline everybody has is **linear interpolation in time per link**: it
draws a straight line through a gap. That is fine in free flow and wrong exactly where it matters, at the edge of a jam.

In [ ]:
show('sth3d_schema.png', 1100)

## 2. Why a low-rank matrix of patches

A travelling wave `f(x − c·t)` is, locally, a sum of a few exponentials in space and time. A signal that is a sum of *r* (damped) exponentials has a **Hankel matrix of rank r**
(the classical Prony / ESPRIT fact; this is also what 'Laplace' gets you: complex frequencies σ + iω are damped waves). So instead of asking the whole
day to be low rank - jams of different sizes make it full rank - we cut it into **overlapping 5-link × 12-step patches**, write each patch as one row, and ask the
*matrix of patches* to be low rank (rows are strongly repeated: the same wave pattern appears at many places and times). Missing cells are the unknowns; observed cells are fixed.
This is solved by ADMM with one **truncated** nuclear norm: keep the *r* largest singular values untouched, shrink the rest.

Two implementation choices that matter and are easy to get wrong:
* the **order of the links** must be the road order, otherwise the 5 'neighbours' of a patch are not neighbours (the LRTC-TNN of the same family does not need this);
* the truncated SVD of the long N×60 matrix is done through the **60×60 Gram matrix** (`eigh` in float64): same result as the SVD (relative difference 1.4e-6, tested below), much faster.

In [ ]:
import numpy as np, torch, torch.nn.functional as Fn
TS, TT = 5, 12                                    # patch: 5 neighbouring links x 12 time steps (= 60 minutes)
def unfold(X, ts=TS, tt=TT):                      # (K, L, T) -> (K, N, ts*tt): every ts x tt window of the space-time matrix becomes one row (Hankel structure)
    return Fn.unfold(X[:, None], (ts, tt)).transpose(1, 2)
def counts(L, T, ts=TS, tt=TT):                   # how many patches cover each cell (needed to average the overlaps back)
    return Fn.fold(Fn.unfold(torch.ones(1, 1, L, T), (ts, tt)), (L, T), (ts, tt))[0, 0]
def fold(P, L, T, cnt, ts=TS, tt=TT):             # (K, N, ts*tt) -> (K, L, T): least-squares inverse of unfold = average of overlapping patches
    return Fn.fold(P.transpose(1, 2), (L, T), (ts, tt))[:, 0] / cnt
def tsvt(M, r, tau):                              # truncated singular-value thresholding: keep the r largest, shrink the rest by tau; via the small Gram matrix (float64)
    G = M.double().T @ M.double(); ev, V = torch.linalg.eigh(G); ev = torch.clamp(ev.flip(0), min=0); V = V.flip(1); s = torch.sqrt(ev); s2 = s.clone(); s2[r:] = torch.clamp(s2[r:] - tau, min=0)
    ok = s > 1e-12 * s[0].clamp(min=1e-300); f = torch.where(ok, s2 / s.clamp(min=1e-300), torch.zeros_like(s)); f[:r] = torch.where(ok[:r], torch.ones_like(f[:r]), torch.zeros_like(f[:r]))
    return (M.double() @ ((V * f) @ V.T)).to(M.dtype)
def tsvt_svd(M, r, tau):                          # reference implementation (slow on long matrices) used by the self-test
    U, s, Vh = torch.linalg.svd(M, full_matrices=False); s = s.clone(); s[r:] = torch.clamp(s[r:] - tau, min=0); return (U * s) @ Vh
def admm(Y, obs, init, F, Finv, theta, max_it=100, eps=1e-4, rho0=1e-5, rho_max=1e5):
    """ADMM for  min ||H(X)||_truncated-nuclear  s.t. X = Y on the observed cells.   F = Hankel map, Finv = its inverse; Y, obs, init share one shape."""
    X = torch.where(obs, Y, init); M = F(X); Lam = torch.zeros_like(M); r = max(1, int(np.ceil(theta * min(M.shape)))); rho = rho0; prev = X.clone(); nY = torch.linalg.norm(Y[obs]) + 1e-12
    for it in range(max_it):
        rho = min(1.05 * rho, rho_max); W = tsvt(F(X) + Lam / rho, r, 1.0 / rho); X = torch.where(obs, Y, Finv(W - Lam / rho)); Lam = Lam + rho * (F(X) - W)
        d = float(torch.linalg.norm(X - prev) / nY); prev = X.clone()
        if d < eps: break
    return X, r, it + 1
def sth_day(Y, obs, init, theta=0.2, **kw):       # A: one space-time matrix (links x time) per day
    L, T = Y.shape; cnt = counts(L, T)
    X, r, it = admm(Y[None], obs[None], init[None], lambda X: unfold(X)[0], lambda M: fold(M[None], L, T, cnt), theta, **kw); return X[0], r, it
def sth_weekday(Y, obs, init, theta=0.1, **kw):   # C: K days of the SAME weekday side by side -> (N, K*60): time x position x day-of-week share one low-rank structure
    K, L, T = Y.shape; cnt = counts(L, T)
    F = lambda X: unfold(X).permute(1, 0, 2).reshape(-1, K * TS * TT); Finv = lambda M: fold(M.reshape(-1, K, TS * TT).permute(1, 0, 2), L, T, cnt)
    return admm(Y, obs, init, F, Finv, theta, **kw)
def pd_interp(B):                                 # linear interpolation in time per link (the baseline every method is compared with)
    import pandas as pd
    out = pd.DataFrame(B.T).interpolate(limit_direction='both').to_numpy().T; return np.where(np.isfinite(out), out, np.nanmean(B))
def selftest():
    torch.manual_seed(0); np.random.seed(0); L, T = 40, 200; res = []
    X = torch.randn(1, L, T, dtype=torch.float64); H = unfold(X); c = counts(L, T).double(); res.append(('fold(unfold(X)) = X', float((fold(H, L, T, c) - X).abs().max()) < 1e-12))
    Y = torch.randn(L, T); o = torch.ones(L, T, dtype=torch.bool); Xf, r, it = sth_day(Y, o, Y.clone()); res.append(('everything observed -> unchanged', float((Xf - Y).abs().max()) < 1e-6))
    x = torch.arange(L).float()[:, None]; t = torch.arange(T).float()[None, :]; F0 = torch.sin(2 * np.pi * t / 48 + 0.15 * x) + 0.6 * torch.sin(2 * np.pi * t / 30 - 0.1 * x) + 0.05 * torch.randn(L, T)   # two travelling waves = low rank
    o = torch.ones(L, T, dtype=torch.bool)
    for _ in range(8): a, b = np.random.randint(0, L - 8), np.random.randint(0, T - 30); o[a:a + 8, b:b + 30] = False
    o &= torch.rand(L, T) > 0.25; B = np.where(o.numpy(), F0.numpy(), np.nan); init = torch.tensor(pd_interp(B), dtype=torch.float32); Xs, r, it = sth_day(torch.where(o, F0, torch.zeros(1)), o, init, theta=0.1, max_it=200); m = ~o
    rmse = lambda A: float(torch.sqrt(torch.mean((A[m] - F0[m]) ** 2))); res.append((f'low-rank wave field, {100 * float(m.float().mean()):.0f}% blank: STH {rmse(Xs):.3f} < interpolation {rmse(init):.3f}', rmse(Xs) < rmse(init)))
    Zt = torch.randn(3000, 60) @ torch.randn(60, 60).tril(); ref = tsvt_svd(Zt, 6, 3.0); e = float((tsvt(Zt, 6, 3.0) - ref).abs().max() / ref.abs().max()); res.append((f'Gram-eigen TSVT == SVD TSVT (rel. {e:.1e})', e < 1e-4))
    for n, ok in res: print(('OK    ' if ok else 'FAIL  ') + n)
    print(f'{sum(ok for _, ok in res)}/{len(res)} OK'); return all(ok for _, ok in res)

**Self-test** - four checks, each one a property the code must have if it is right (a Hankel round-trip, an identity case, a low-rank wave field with 38 % block gaps where STH must beat interpolation, and the Gram-eigen = SVD equality):

In [ ]:
selftest()

## 3. The third mode: the same weekday on K different days

The paper (and our first implementation) completes **one space-time matrix per day**. But a road has a weekly rhythm: Tuesday at 08:00 looks like last Tuesday at 08:00.
If the same patch at the same time and place on K equal weekdays is put **side by side** (an N × (K·60) matrix instead of N × 60), then *position × time × day-of-week* share one low-rank basis,
and a gap on one Tuesday can borrow from the other Tuesdays. A fourth variant, all days stacked **under** each other (one shared basis), was measured too: its errors correlate 0.99 with the per-day version, i.e. it adds nothing, so it is not shown.

* **A** - STH per day (θ 0.2, rank 12);
* **C** - STH weekday-3-D (θ 0.1, rank 30 for K = 5): `sth_weekday` above.

θ is the rank fraction; both values were chosen earlier on a *different* set of pseudo-gaps (validation month), not tuned here.

## 4. Live test on real corridor-days

Five Tuesdays of D12_I5_S (Jan–Feb 2031, train layer). In each day, ten blocks of **8 links × 90 minutes** are hidden from every method (only cells that were really observed are hidden, so there is a
truth to score against). Four different random blockings. The score is the speed RMSE on the hidden cells, in km/h.

In [ ]:
import time
panel = 'D12_I5_S'; dates = ['2031-01-14', '2031-01-21', '2031-01-28', '2031-02-04', '2031-02-11']        # five Tuesdays
c = corridor(panel); L = len(c['order']); GT = np.stack([day_grid(panel, d, c['order']) for d in dates]).transpose(0, 2, 1)     # (K, L, T) eligible cells, NaN elsewhere
def run(seed):
    rng = np.random.default_rng(seed); TEST = np.zeros(GT.shape, bool); blocks = []
    for k in range(len(dates)):
        for _ in range(10):                                     # ten blank blocks per day: 8 neighbouring links x 18 steps (90 min), like the task's blanked bands
            a, b = rng.integers(0, L - 8), rng.integers(0, 288 - 18); TEST[k, a:a + 8, b:b + 18] = True; blocks.append((k, int(a), int(b)))
    TEST &= np.isfinite(GT); O = np.isfinite(GT) & ~TEST        # we hide eligible cells; the cells that were never eligible are unknown anyway
    INIT = np.stack([pd_interp(np.where(O[k], GT[k], np.nan)) for k in range(len(dates))])
    Y = torch.tensor(np.where(O, GT, 0.0), dtype=torch.float32); Ot = torch.tensor(O); It = torch.tensor(INIT, dtype=torch.float32)
    rmse = lambda A: float(np.sqrt(np.mean((A[TEST] - GT[TEST]) ** 2)))
    FA = np.stack([sth_day(Y[k], Ot[k], It[k], theta=0.2)[0].numpy() for k in range(len(dates))]); FC = sth_weekday(Y, Ot, It, theta=0.1)[0].numpy()
    return dict(seed=seed, hidden=int(TEST.sum()), interp=rmse(INIT), A=rmse(FA), C=rmse(FC), mean_AC=rmse((FA + FC) / 2), corr=float(np.corrcoef((FA - GT)[TEST], (FC - GT)[TEST])[0, 1])), (INIT, FA, FC, TEST, blocks)
t0 = time.time(); rows = []
for s in range(4): r, last = run(s); rows.append(r)
R = pd.DataFrame(rows).set_index('seed').round(3); R.loc['mean'] = R.mean().round(3); print(R.to_string()); print(f'\n{time.time() - t0:.0f} s;  C vs A: {100 * (R.loc["mean", "C"] / R.loc["mean", "A"] - 1):+.1f}%   A vs interpolation: {100 * (R.loc["mean", "A"] / R.loc["mean", "interp"] - 1):+.1f}%')

Read it with care: **4 blockings is a small sample.** The weekday-3-D variant is better than the per-day version in 3 of 4; in blocking 1 the per-day version wins. Their errors correlate 0.6–0.8; the *average* of A and C is better than both only in that one blocking and worse than C alone in the other three. The size of the errors varies a lot between blockings (interpolation: 5.0 to 9.1 km/h), i.e. with where the hidden blocks happen to fall relative to the jams.

The figure below is a **deliberately hard case**: the hidden block (dashed) where plain interpolation is worst. A jam clears inside the block; interpolation draws a slow diagonal, the per-day STH
partly keeps the old shape, and the weekday-3-D version recovers the sharp dissolve of the jam, because the other Tuesdays have seen it.

In [ ]:
show('sth3d_example.png', 1150)

## 5. The fair comparison: the validation month, six groups of pseudo-bands

The block above is picked. The table below is not: **pseudo-bands** cut out of the *observed* cells of the validation month (so the truth is known), 3 groups per corridor, the same hidden cells for every method.
`TNN` is the older LRTC-TNN (links × days × slots, no Hankel structure, no road order needed). Numbers are parsed from the raw logs of our stage-0 runs (`t1_band_sth3d_stage0.py` / `…save_*.log`; the logs are not included here, the figure's source table is in the figure dataset).

In [ ]:
show('sth3d_vrmse.png', 1000)

In [ ]:
pd.read_csv(next(KIN.glob('**/sth3d_vrmse_table.csv')), index_col=0)

* Best of the four per group: **TNN in 3** (D12_I5_S g1, g2; D7_I10_E g2), **C in 2** (D7_I10_E g0, g1), **A in 1** (D12_I5_S g0).
* **C beats A in 4 of 6 groups** (all three on D7_I10_E and D12_I5_S g1) and loses to it in two (D12_I5_S g0: 12.60 vs 12.10; g2: 9.90 vs 9.61).
* TNN is also **worse than plain interpolation in two groups** (D12_I5_S g0: 17.96 vs 12.92; D7_I10_E g0: 9.06 vs 7.84). A fill that is sometimes much worse than the baseline is dangerous alone and useful in a stack.
* The *average* of A and C (not in the table; from the same logs, θ 0.1) beats both only in 2 of 6 groups, both on D12_I5_S (g0 11.70, g2 9.25): the two share most of their errors (correlation 0.75–0.85).

Not measured here: speed only (flow was not part of this test), 2 of the 7 corridors, one validation month, ADMM stopped at its 100-iteration cap (it has not converged - this is the method *as it runs*).

## 6. What it was worth on the leaderboard (static numbers from our log)

The fills are not an answer; they are **inputs**. In our submission a gradient-boosted band model (LightGBM) gets, for every blanked cell, the edge frames of the band, ~47 features of the neighbourhood, and the fills of LRTC-TNN,
an LMMSE interpolator, STH per day, and STH weekday-3-D. Public leaderboard (Task-1 weight 0.35), same queue and ODME layers throughout:

| submission | what changed | public score |
|---|---|---|
| sub38 | base model | 0.88682 |
| sub54 | + band model with LRTC / LMMSE / STH-per-day columns | 0.88705 |
| sub57 | + 3 more corridors + the 3-D-STH columns | 0.88723 |
| **sub59** | **the pure 3-D-STH fill as the answer in the same cells, no model** | **0.88648** |

* sub59 is a *diagnostic probe*: replace the model's answer in 120,297 cells by the 3-D fill alone. Predicted before uploading from local numbers (pure 3-D fill vs the base model, @2 % error tolerance × the 0.35 task weight): **−0.00035**. Measured: **−0.00034** (0.88682 → 0.88648). With full replacement the local-to-board transfer is ≈ 1 - where
  the model only blends a fill in (λ = 0.5–0.75) we saw only a quarter to a half of the local gain.
* So the 3-D fill is the best *single* fill in the pooled evaluation over eight corridors (1.3 M pseudo-band cells; per group, section 5, TNN wins some) and still **worse than what the base model already had** in these cells. Its value in sub57 is small: +0.00008 local (@2 %) for the four 3-D columns, and the sub57 gain over sub55 (+0.00014 on the board) mixes those columns with three new corridors, which we did not upload separately. **The two cannot be separated on the board.**
* The best stand-alone component is not the best contribution to a model. The model can use what the fills *disagree about* (the error correlation of A and C is 0.6–0.85, so they do carry different information) - but how much that is worth is small: +0.00008 here.

Not measured: a λ-blend of the pure fill with the base model on the board; the private-set score (hidden until the end of the competition).

## 7. Take-aways
1. A paper method becomes trustworthy here because of the **self-test**: four properties it must have, one of them an identity that fails loudly if the Hankel inverse is off by an index.
2. The third mode is a real but uneven improvement as a *fill* (−21 % vs per-day in the live test, better than per-day in 4 of 6 validation groups) and a tiny one as a *feature*.
3. **Predict the board before you upload, from local numbers, and write the prediction down.** Here the proxy held for a large unblended replacement and broke (too pessimistic) for the blended steps.

*Code: `sth_core` above (≈ 60 lines); the derivation of every number is in the project's measurement log. Data: the competition release only. Written with Claude Code.*